# AACA - Fine-tuning OCR pour formules mathématiques manuscrites

Ce notebook montre une démarche complète pour tester puis fine-tuner un modèle **image → LaTeX** afin de reconnaître des formules mathématiques manuscrites.

L'objectif est d'obtenir un module utilisable dans AACA pour transformer une image de formule manuscrite en code LaTeX, par exemple :

```text
image manuscrite → \int_0^1 x^2 dx
```

Le notebook suit cette logique :

1. Préparer l'environnement Colab.
2. Monter Google Drive.
3. Préparer le dataset image + LaTeX.
4. Tester le modèle pré-entraîné.
5. Préparer les fichiers d'entraînement.
6. Lancer le fine-tuning.
7. Évaluer le modèle.
8. Exporter le modèle pour l'intégrer dans AACA.

> Important : le fine-tuning dépend de la version exacte de `pix2tex`. Si une commande change, l'idée reste la même : dataset d'images + labels LaTeX, configuration d'entraînement, puis export du checkpoint.

## 1. Activer le GPU

Dans Google Colab :

```text
Runtime → Change runtime type → GPU
```

Un GPU T4 suffit pour commencer avec un petit dataset. Pour un projet PFA, un dataset de 300 à 1000 formules est déjà intéressant.

In [2]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


## 2. Monter Google Drive

On stocke le dataset et les checkpoints dans Google Drive pour éviter de tout perdre quand la session Colab s'arrête.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

## 3. Installer les dépendances

On installe `pix2tex` avec les dépendances d'entraînement. On ajoute aussi quelques bibliothèques utiles pour manipuler les images et évaluer les prédictions.

In [ ]:
!pip install -q "pix2tex[train]" jiwer python-Levenshtein pandas pillow matplotlib opencv-python

In [ ]:
import os
import re
import json
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

BASE_DIR = Path('/content/drive/MyDrive/AACA_math_dataset')
TRAIN_IMG_DIR = BASE_DIR / 'train_images'
VAL_IMG_DIR = BASE_DIR / 'val_images'
TEST_IMG_DIR = BASE_DIR / 'test_images'

TRAIN_CSV = BASE_DIR / 'train_labels.csv'
VAL_CSV = BASE_DIR / 'val_labels.csv'
TEST_CSV = BASE_DIR / 'test_labels.csv'

for d in [BASE_DIR, TRAIN_IMG_DIR, VAL_IMG_DIR, TEST_IMG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Dossier dataset :', BASE_DIR)

## 4. Structure attendue du dataset

Dans Google Drive, prépare cette structure :

```text
AACA_math_dataset/
  train_images/
    formula_0001.png
    formula_0002.png
  val_images/
    formula_0101.png
  test_images/
    formula_0201.png

  train_labels.csv
  val_labels.csv
  test_labels.csv
```

Chaque CSV doit contenir deux colonnes :

```csv
filename,latex
formula_0001.png,\int_0^1 x^2 dx
formula_0002.png,\sum_{i=1}^{n} i^2
```

Conseil : commence avec des formules simples : intégrales, sommes, limites, fractions, racines, dérivées, matrices simples.

## 5. Créer un petit dataset exemple

Cette cellule crée un CSV exemple si tu n'as pas encore préparé ton dataset. Elle ne crée pas de vraies images manuscrites. Tu dois remplacer les fichiers images par tes propres captures.

In [ ]:
if not TRAIN_CSV.exists():
    example = pd.DataFrame([
        {'filename': 'formula_0001.png', 'latex': r'\int_0^1 x^2 dx'},
        {'filename': 'formula_0002.png', 'latex': r'\sum_{i=1}^{n} i^2'},
        {'filename': 'formula_0003.png', 'latex': r'\frac{x+1}{x-1}'},
        {'filename': 'formula_0004.png', 'latex': r'\lim_{x \to 0} \frac{\sin x}{x}'},
    ])
    example.to_csv(TRAIN_CSV, index=False)
    example.iloc[:2].to_csv(VAL_CSV, index=False)
    example.iloc[2:].to_csv(TEST_CSV, index=False)
    print('CSV exemples créés. Remplace maintenant les images par tes vraies images manuscrites.')
else:
    print('CSV déjà existant :', TRAIN_CSV)

## 6. Vérifier le dataset

Cette étape vérifie que chaque ligne du CSV possède une image correspondante.

In [ ]:
def check_dataset(csv_path, image_dir):
    df = pd.read_csv(csv_path)
    required = {'filename', 'latex'}
    if not required.issubset(df.columns):
        raise ValueError(f'{csv_path} doit contenir les colonnes filename et latex')
    missing = []
    for name in df['filename']:
        if not (image_dir / name).exists():
            missing.append(name)
    print(csv_path.name, '=>', len(df), 'lignes')
    if missing:
        print('Images manquantes :', missing[:10])
    else:
        print('OK : toutes les images existent')
    return df

train_df = check_dataset(TRAIN_CSV, TRAIN_IMG_DIR)
val_df = check_dataset(VAL_CSV, VAL_IMG_DIR)
test_df = check_dataset(TEST_CSV, TEST_IMG_DIR)

## 7. Visualiser quelques exemples

Avant de fine-tuner, vérifie visuellement que les images sont bien recadrées et que le LaTeX correspond exactement à l'image.

In [ ]:
def show_samples(df, image_dir, n=4):
    n = min(n, len(df))
    if n == 0:
        print('Aucun exemple à afficher')
        return
    plt.figure(figsize=(12, 3 * n))
    for i in range(n):
        row = df.iloc[i]
        path = image_dir / row['filename']
        plt.subplot(n, 1, i + 1)
        if path.exists():
            img = Image.open(path).convert('RGB')
            plt.imshow(img)
        plt.axis('off')
        plt.title(row['latex'])
    plt.tight_layout()

show_samples(train_df, TRAIN_IMG_DIR, n=4)

## 8. Prétraiter les images

Le prétraitement aide beaucoup pour les formules manuscrites : recadrage, contraste, fond blanc, suppression du bruit. Cette cellule crée des copies prétraitées dans de nouveaux dossiers.

Tu peux l'utiliser ou la modifier selon la qualité de tes images.

In [ ]:
import cv2
import numpy as np

PREP_DIR = BASE_DIR / 'preprocessed'
PREP_TRAIN_IMG_DIR = PREP_DIR / 'train_images'
PREP_VAL_IMG_DIR = PREP_DIR / 'val_images'
PREP_TEST_IMG_DIR = PREP_DIR / 'test_images'
for d in [PREP_TRAIN_IMG_DIR, PREP_VAL_IMG_DIR, PREP_TEST_IMG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def preprocess_formula_image(src_path, dst_path):
    img = cv2.imread(str(src_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False
    # Amélioration du contraste
    img = cv2.equalizeHist(img)
    # Binarisation adaptative
    th = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY, 31, 15)
    # Recadrage sur les pixels non blancs
    coords = cv2.findNonZero(255 - th)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = 12
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(th.shape[1], x + w + pad), min(th.shape[0], y + h + pad)
        th = th[y0:y1, x0:x1]
    # Sauvegarde
    cv2.imwrite(str(dst_path), th)
    return True

def preprocess_split(df, src_dir, dst_dir):
    ok, fail = 0, 0
    for _, row in df.iterrows():
        src = src_dir / row['filename']
        dst = dst_dir / row['filename']
        if src.exists() and preprocess_formula_image(src, dst):
            ok += 1
        else:
            fail += 1
    print(dst_dir.name, 'prétraitées :', ok, 'échecs :', fail)

preprocess_split(train_df, TRAIN_IMG_DIR, PREP_TRAIN_IMG_DIR)
preprocess_split(val_df, VAL_IMG_DIR, PREP_VAL_IMG_DIR)
preprocess_split(test_df, TEST_IMG_DIR, PREP_TEST_IMG_DIR)

## 9. Tester le modèle pré-entraîné

Avant de faire le fine-tuning, on teste le modèle pré-entraîné. Cela permet de créer une baseline : on comparera ensuite avant/après fine-tuning.

In [ ]:
from pix2tex.cli import LatexOCR

pretrained_model = LatexOCR()

def predict_one(model, image_path):
    img = Image.open(image_path).convert('RGB')
    return model(img)

if len(test_df) > 0:
    row = test_df.iloc[0]
    image_path = PREP_TEST_IMG_DIR / row['filename']
    if not image_path.exists():
        image_path = TEST_IMG_DIR / row['filename']
    if image_path.exists():
        pred = predict_one(pretrained_model, image_path)
        print('Vérité :', row['latex'])
        print('Prédit :', pred)
    else:
        print('Ajoute d’abord des images de test.')

## 10. Évaluer rapidement la baseline

On mesure ici un score simple : exact match et distance d'édition. Ce n'est pas une évaluation mathématique parfaite, mais c'est utile pour comparer deux modèles.

In [ ]:
from Levenshtein import distance as levenshtein_distance

def normalize_latex(s):
    s = str(s).strip()
    s = re.sub(r'\s+', '', s)
    return s

def evaluate_model(model, df, image_dir, max_items=None):
    rows = []
    subset = df if max_items is None else df.head(max_items)
    for _, row in subset.iterrows():
        image_path = image_dir / row['filename']
        if not image_path.exists():
            continue
        truth = row['latex']
        pred = predict_one(model, image_path)
        nt, npred = normalize_latex(truth), normalize_latex(pred)
        rows.append({
            'filename': row['filename'],
            'truth': truth,
            'prediction': pred,
            'exact': int(nt == npred),
            'edit_distance': levenshtein_distance(nt, npred)
        })
    result = pd.DataFrame(rows)
    if len(result):
        print('Exact match :', result['exact'].mean())
        print('Distance moyenne :', result['edit_distance'].mean())
    return result

baseline_results = evaluate_model(pretrained_model, test_df, PREP_TEST_IMG_DIR, max_items=20)
baseline_results.head()

## 11. Convertir les CSV au format attendu par pix2tex

`pix2tex` peut générer des fichiers `.pkl` à partir d'un dossier d'images et d'un fichier texte contenant les équations. Pour éviter les erreurs d'ordre, on crée des dossiers propres avec des noms triés et un fichier `labels.txt` correspondant.

In [ ]:
P2T_DIR = BASE_DIR / 'pix2tex_ready'
P2T_TRAIN_IMG_DIR = P2T_DIR / 'train_images'
P2T_VAL_IMG_DIR = P2T_DIR / 'val_images'
P2T_TEST_IMG_DIR = P2T_DIR / 'test_images'
for d in [P2T_TRAIN_IMG_DIR, P2T_VAL_IMG_DIR, P2T_TEST_IMG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def prepare_pix2tex_split(df, src_dir, dst_dir, labels_txt):
    df = df.sort_values('filename').reset_index(drop=True)
    equations = []
    copied = 0
    for i, row in df.iterrows():
        src = src_dir / row['filename']
        if not src.exists():
            continue
        ext = Path(row['filename']).suffix.lower() or '.png'
        new_name = f'{i:06d}{ext}'
        shutil.copy(src, dst_dir / new_name)
        equations.append(str(row['latex']).strip())
        copied += 1
    labels_txt.write_text('\n'.join(equations), encoding='utf-8')
    print(labels_txt.name, '=>', copied, 'images')

prepare_pix2tex_split(train_df, PREP_TRAIN_IMG_DIR, P2T_TRAIN_IMG_DIR, P2T_DIR / 'train_labels.txt')
prepare_pix2tex_split(val_df, PREP_VAL_IMG_DIR, P2T_VAL_IMG_DIR, P2T_DIR / 'val_labels.txt')
prepare_pix2tex_split(test_df, PREP_TEST_IMG_DIR, P2T_TEST_IMG_DIR, P2T_DIR / 'test_labels.txt')

## 12. Générer les fichiers `.pkl`

Ces fichiers sont utilisés par l'entraînement de `pix2tex`.

Si cette commande échoue à cause d'une différence de version, consulte l'aide :

```bash
python -m pix2tex.dataset.dataset --help
```

In [ ]:
!python -m pix2tex.dataset.dataset \
  --equations "{P2T_DIR / 'train_labels.txt'}" \
  --images "{P2T_TRAIN_IMG_DIR}" \
  --out "{P2T_DIR / 'train.pkl'}"

!python -m pix2tex.dataset.dataset \
  --equations "{P2T_DIR / 'val_labels.txt'}" \
  --images "{P2T_VAL_IMG_DIR}" \
  --out "{P2T_DIR / 'val.pkl'}"

## 13. Créer une configuration d'entraînement

Le fichier de configuration exact peut varier selon la version de `pix2tex`. Cette cellule crée une configuration minimale à adapter. Si ton installation contient un fichier config officiel, il est préférable de le copier et de modifier uniquement `data`, `valdata`, `epochs` et `batchsize`.

In [ ]:
CONFIG_PATH = P2T_DIR / 'aaca_finetune_config.yaml'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

config_text = f'''
# Configuration indicative pour fine-tuning pix2tex.
# Selon la version de pix2tex, certains champs peuvent devoir être ajustés.

data: {P2T_DIR / 'train.pkl'}
valdata: {P2T_DIR / 'val.pkl'}
save_path: {CHECKPOINT_DIR}

epochs: 5
batchsize: 4
lr: 0.00001
num_workers: 2
'''

CONFIG_PATH.write_text(config_text, encoding='utf-8')
print(CONFIG_PATH)
print(CONFIG_PATH.read_text())

## 14. Lancer le fine-tuning

Cette étape peut prendre du temps. Commence avec peu d'epochs, par exemple 3 à 5, puis augmente si les résultats sont prometteurs.

Si Colab affiche une erreur de mémoire GPU :

- diminue `batchsize` à 2 ou 1 ;
- réduis la taille des images ;
- utilise un modèle plus petit ;
- limite le dataset pour un premier test.

In [ ]:
!python -m pix2tex.train --config "{CONFIG_PATH}"

## 15. Trouver les checkpoints générés

Après l'entraînement, vérifie les fichiers sauvegardés. Le meilleur checkpoint sera utilisé dans AACA.

In [ ]:
for path in CHECKPOINT_DIR.rglob('*'):
    print(path)

## 16. Tester le modèle fine-tuné

Le chargement exact du checkpoint dépend de la version de `pix2tex`. Si `LatexOCR()` accepte un chemin de checkpoint dans ta version, indique-le ici. Sinon, consulte la documentation de ta version installée.

Même si cette cellule doit être adaptée, l'idée est de comparer :

```text
modèle pré-entraîné vs modèle fine-tuné
```

In [ ]:
# À adapter selon le checkpoint obtenu et la version de pix2tex.
# Exemple conceptuel :
# finetuned_model = LatexOCR(checkpoint=str(CHECKPOINT_DIR / 'best.pth'))
# finetuned_results = evaluate_model(finetuned_model, test_df, PREP_TEST_IMG_DIR, max_items=50)
# finetuned_results.head()

print('Charge ici le checkpoint fine-tuné selon la version de pix2tex utilisée.')

## 17. Exporter le modèle pour AACA

Quand tu obtiens un checkpoint satisfaisant, copie-le dans un dossier exportable :

In [ ]:
EXPORT_DIR = BASE_DIR / 'export_for_aaca'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Exemple : copie manuelle du meilleur checkpoint quand tu connais son nom.
# shutil.copy(CHECKPOINT_DIR / 'best.pth', EXPORT_DIR / 'aaca_math_ocr_best.pth')

print('Dossier export :', EXPORT_DIR)

## 18. Intégration dans le backend AACA

Après export, tu peux ajouter un service dans ton backend :

```text
backend/app/services/math_formula_ocr_service.py
```

Rôle du service :

1. recevoir une image ou un crop de formule ;
2. charger le modèle fine-tuné ;
3. retourner le LaTeX prédit ;
4. analyser le LaTeX pour identifier intégrale, somme, limite, fraction, etc. ;
5. sauvegarder le résultat dans la note.

Exemple de sortie attendue :

```json
{
  "latex": "\\sum_{i=1}^{n} i^2",
  "type": "summation",
  "index": "i",
  "lower_bound": "1",
  "upper_bound": "n",
  "expression": "i^2"
}
```

## 19. Bonnes pratiques pour ton PFA

Pour ton rapport, tu peux présenter cette amélioration comme une perspective ou un module expérimental :

> Le fine-tuning du module d'extraction mathématique consiste à adapter un modèle image-vers-LaTeX pré-entraîné à un corpus de formules manuscrites académiques. Le dataset est composé d'images de formules associées à leur transcription LaTeX. Après prétraitement et augmentation des images, le modèle est réentraîné afin d'améliorer sa précision sur les écritures manuscrites rencontrées dans les cours.

La preuve expérimentale simple :

```text
1. Prendre 50 formules de test.
2. Tester pix2tex pré-entraîné.
3. Fine-tuner avec ton dataset.
4. Tester à nouveau.
5. Comparer exact match et distance d'édition.
```

Même une petite amélioration est intéressante si elle est bien expliquée.

## 20. Résumé final

La méthode complète est :

```text
Collecte de formules manuscrites
→ annotation en LaTeX
→ prétraitement des images
→ test du modèle pré-entraîné
→ fine-tuning
→ évaluation
→ export du checkpoint
→ intégration dans AACA
```

Le point le plus important n'est pas seulement la puissance GPU. Le plus important est d'avoir un dataset propre : image bien recadrée + LaTeX exact.